# 02 — Feature Engineering
Derived features, encoding strategy, train/test split, and SMOTE

In [1]:
import sys, os
# Move to project root so relative paths (data/, outputs/) resolve correctly
# Works whether the kernel starts from notebooks/ or the project root.
_here = os.path.abspath('.')
if os.path.basename(_here) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.path.abspath('.'))
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.loader import load_raw_data
from src.data.cleaner import clean_data
from src.features.engineering import engineer_features, get_feature_columns, SERVICE_COLUMNS
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

## 1. Load & Clean

In [2]:
df_raw = load_raw_data()
df = clean_data(df_raw)
print('Clean shape:', df.shape)

Clean shape: (7043, 20)


## 2. Derived Features — Before / After

In [3]:
print('BEFORE feature engineering:')
print(df[['tenure', 'TotalCharges', 'MonthlyCharges'] + SERVICE_COLUMNS].head(3).to_string())

BEFORE feature engineering:
   tenure  TotalCharges  MonthlyCharges PhoneService     MultipleLines OnlineSecurity OnlineBackup DeviceProtection TechSupport StreamingTV StreamingMovies
0       1         29.85           29.85           No  No phone service             No          Yes               No          No          No              No
1      34       1889.50           56.95          Yes                No            Yes           No              Yes          No          No              No
2       2        108.15           53.85          Yes                No            Yes          Yes               No          No          No              No


In [4]:
df_feat = engineer_features(df)
print('\nAFTER — new columns:')
print(df_feat[['tenure', 'tenure_group', 'total_services',
               'charges_per_month', 'is_high_value']].head(10).to_string())


AFTER — new columns:
   tenure tenure_group  total_services  charges_per_month  is_high_value
0       1        0-12m               1          29.850000              0
1      34       25-48m               3          55.573529              0
2       2        0-12m               3          54.075000              0
3      45       25-48m               3          40.905556              0
4       2        0-12m               1          75.825000              0
5       8        0-12m               5         102.562500              1
6      22       13-24m               4          88.609091              0
7      10        0-12m               1          30.190000              0
8      28       25-48m               6         108.787500              1
9      62         61m+               3          56.257258              0


In [5]:
print('tenure_group distribution:')
print(df_feat['tenure_group'].value_counts().sort_index())
print('\ntotal_services distribution:')
print(df_feat['total_services'].value_counts().sort_index())
print('\nis_high_value:', df_feat['is_high_value'].value_counts().to_dict())

tenure_group distribution:
tenure_group
0-12m     2186
13-24m    1024
25-48m    1594
49-60m     832
61m+      1407
Name: count, dtype: int64

total_services distribution:
total_services
0      80
1    1701
2    1188
3     965
4     922
5     908
6     676
7     395
8     208
Name: count, dtype: int64

is_high_value: {0: 5285, 1: 1758}


## 3. Feature Column Registry

In [6]:
feature_cols = get_feature_columns()
for group, cols in feature_cols.items():
    print(f'{group} ({len(cols)}): {cols}')

numerical (5): ['tenure', 'MonthlyCharges', 'TotalCharges', 'total_services', 'charges_per_month']
binary (1): ['is_high_value']
categorical (17): ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'tenure_group']


## 4. Train / Test Split (80/20 stratified)

In [7]:
all_features = (
    feature_cols['numerical'] +
    feature_cols['binary'] +
    feature_cols['categorical']
)

X = df_feat[all_features]
y = df_feat['Churn']

# One-hot encode categoricals before splitting so SMOTE operates on numerics
X_encoded = pd.get_dummies(X, columns=feature_cols['categorical'], drop_first=True)
print('Encoded feature matrix shape:', X_encoded.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Train churn rate: {y_train.mean():.1%} | Test: {y_test.mean():.1%}')

Encoded feature matrix shape: (7043, 37)
Train: (5634, 37) | Test: (1409, 37)
Train churn rate: 26.5% | Test: 26.5%


## 5. Class Imbalance — Before SMOTE

In [8]:
print('y_train before SMOTE:')
print(y_train.value_counts())
print(f'Positive rate: {y_train.mean():.1%}')

y_train before SMOTE:
Churn
0    4139
1    1495
Name: count, dtype: int64
Positive rate: 26.5%


## 6. SMOTE — Applied Only on Training Set

In [9]:
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print('y_train AFTER SMOTE:')
print(pd.Series(y_train_sm).value_counts())
print(f'Positive rate after SMOTE: {pd.Series(y_train_sm).mean():.1%}')
print(f'Shape change: {X_train.shape} -> {X_train_sm.shape}')

y_train AFTER SMOTE:
Churn
0    4139
1    4139
Name: count, dtype: int64
Positive rate after SMOTE: 50.0%
Shape change: (5634, 37) -> (8278, 37)


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (y_plot, title) in zip(axes, [
    (y_train,    'Before SMOTE'),
    (pd.Series(y_train_sm), 'After SMOTE'),
]):
    counts = y_plot.value_counts().sort_index()
    ax.bar(['No Churn', 'Churn'], counts.values, color=['#0173b2', '#de8f05'])
    ax.set_title(title)
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 20, f'{v:,}', ha='center')
fig.suptitle('Class Imbalance Before vs After SMOTE (training set only)', fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/smote_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Final Feature Matrix

In [11]:
print(f'X_train (post-SMOTE): {X_train_sm.shape}')
print(f'X_test             : {X_test.shape}')
print(f'y_train            : {len(y_train_sm)}')
print(f'y_test             : {len(y_test)}')
print(f'\nFeature names ({len(X_encoded.columns)}):')
print(list(X_encoded.columns))

X_train (post-SMOTE): (8278, 37)
X_test             : (1409, 37)
y_train            : 8278
y_test             : 1409

Feature names (37):
['tenure', 'MonthlyCharges', 'TotalCharges', 'total_services', 'charges_per_month', 'is_high_value', 'gender_Male', 'SeniorCitizen_Yes', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check', 'tenure_group_13-24m', 'tenure_group_25-48m

In [12]:
# Persist split for downstream notebooks
import joblib
from pathlib import Path
Path('outputs/reports').mkdir(exist_ok=True)
joblib.dump({
    'X_train': X_train_sm, 'y_train': y_train_sm,
    'X_test':  X_test,     'y_test':  y_test,
    'feature_names': list(X_encoded.columns),
}, 'outputs/reports/train_test_split.joblib')
print('Split saved to outputs/reports/train_test_split.joblib')

Split saved to outputs/reports/train_test_split.joblib
